# LSTMs and Transformers for Word Sense Disambiguation

**Nikolai Ilinykh, Adam Ek, Simon Dobnik, and others. Last modified by Felix Morger.**

The lab is an exploration and learning exercise to be done in a group and also in discussion with the teachers and other students.

Before starting, please read the instructions on how to work in groups on Canvas.

Write all your answers and the code in the appropriate boxes below.

Static distributional vectors are not trained to distinguish between different *word senses*. They learn all senses per token or word. We continue our exploration of word vectors by considering *trainable vectors* or *word embeddings* for Word Sense Disambiguation (WSD) that include semantic information from the contexts in which a word occurs. We experiment with and compare LSTMs and transformer models, e.g. BERT. The purpose of this exercise is also to learn how to use vector representations from neural models in a downstream task of word sense disambiguation.

**Dependencies**

* Pytorch
    * Installation instructions: https://pytorch.org/
    * Tutorials: https://pytorch.org/tutorials/beginner/basics/intro.html
    * Some useful basic operations: https://jhui.github.io/2018/02/09/PyTorch-Basic-operations
* BERT
    * [HuggingFace transformers](https://huggingface.co/docs/transformers/en/index)
    * [BERT model (example)](https://huggingface.co/google-bert/bert-base-uncased)

**Running the code**

As we are learning about the models, and also what methods work and do not work for our semantic tasks, we are not interested in achieving a state-of-the-art performance. We are learning about different implementations and differences in performance in different conditions.

**On using generative AI for this assignment:** For this lab, the use of generative AI is permitted as a supporting tool, provided it is done in a responsible and conscious manner and that you state clearly with each question how it was used. However, generative AI must never replace the intellectual work you are expected to carry out. Note that the purpose of this lab is to learn some basic coding of the main neural architectures used in natural language processing. You are responsible for ensuring that such tools are used in a way that supports the development of the skills the module is designed to promote. It is your responsibility to ensure that submitted work is the result of independent intellectual effort.

**Getting help:** We encourage you to use Canvas discussions to post questions and interact with teachers and also each other. Provide useful tips, but of course do not reveal the exact answer across the groups as each group should should work out their own solutions. Remember that in most cases there is also not a single correct answer and implementations may differ.


## Word Sense Disambiguation Task

The goal of word sense disambiguation is to train a model to find the sense of a word (homonyms of a word-form). For example, the word "bank" can mean "sloping land" or "financial institution". 

(a) "I deposited my money in the **bank**" (financial institution)

(b) "I swam from the river **bank**" (sloping land)

In case a) and b), we can determine the meaning of "bank" based on the *context*. To utilize context in a semantic model, we use *contextualized word representations*.

Previously, we worked with *static word representations*, i.e., the representation does not depend on the context. To illustrate, we can consider sentences (a) and (b), where the word **bank** would have the same static representation in both sentences, which means that it becomes difficult for us to predict its sense. What we want is to create representations that depend on the context, i.e., *contextualized embeddings*.

As we have discussed in the class, contextualized representations can come in the form of pre-training the model for some "general" task and then fine-tuning it for some downstream task. Here we will do the following:

(1) Train and test LSTM model directly for word sense disambiguation. We will learn contextualized representations within this model.

(2) Take BERT that was pre-trained on masked language modeling and next sentence prediction. Fine-tune it on our data and test it for the word sense disambiguation on the task dataset. The idea for you is to explore how pre-trained contextualized representations from BERT can be updated and used for the downstream task of word sense disambiguation.

Your overall task in this lab is to create a neural network model that can disambiguate the word sense of 30 different words.

In [1]:
%pip install -r ./packages.txt


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# first we import some packages that we need

# here add any package that you will need later

import torch
import torch.nn as nn

# our hyperparameters (add more when/if you need them)
device = torch.device('cuda:0')

batch_size = 8
learning_rate = 0.001
epochs = 3

# 1. Working with Data

A central part of any machine learning system is the data we're working with.

In this section, we will split the data (the dataset is in `wsd_data.txt`) into a training set and a test set.


## Data

The dataset we will use contains different word senses for 30 different words. The data is organized as follows (values separated by tabs), where each line is a separate item in the dataset:

- Column 1: word-sense, e.g., keep%2:42:07::
- Column 2: word-form, e.g., keep.v
- Column 3: index of word, e.g., 15
- Column 4: white-space tokenized context, e.g., Action by the Committee In pursuance of its mandate , the Committee will continue to keep under review the situation relating to the question of Palestine and participate in relevant meetings of the General Assembly and the Security Council . The Committee will also continue to monitor the situation on the ground and draw the attention of the international community to urgent developments in the Occupied Palestinian Territory , including East Jerusalem , requiring international action .


### Splitting the Data

Your first task is to separate the data into a *training set* and a *test set*.

The training set should contain 80% of the examples, and the test set the remaining 20%.

The examples for the test/training set should be selected **randomly**.

Save each dataset into a .csv file for loading later.

**[2 marks]**

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
def data_split(dataset_path):
    df = pd.read_csv(dataset_path, sep ="\t", header= None)

    train_split, test_split = train_test_split(
        df, train_size =0.8, 
        test_size = 0.2, 
        shuffle = True, 
        random_state = 42)
    # your code goes here
    train_split.to_csv("train.csv", index = False)
    test_split.to_csv("test.csv", index = False)
    return train_split, test_split
#data_split("wsd_data.txt")


In [5]:
train_data, test_data = data_split('./wsd_data.txt')

print(f"Total data size: {len(train_data) + len(test_data)}")
print(f"Training data size: {len(train_data)}")
print(f"Testing data size: {len(test_data)}")

Total data size: 76049
Training data size: 60839
Testing data size: 15210


In [6]:
print(train_data.columns)
print(train_data.head())

Index([0, 1, 2, 3], dtype='int64')
                                 0          1   2  \
16323             extend%2:30:02::   extend.v  51   
59400             active%3:00:03::   active.a  38   
32777              serve%2:42:01::    serve.v  48   
63174               case%1:09:00::     case.n  40   
30371  regular%5:00:00:standard:02  regular.a  66   

                                                       3  
16323  ( vii ) BGL - General Trust Fund for the Core ...  
59400  Romania recommended that the Government take a...  
32777  The committee could serve as the key point of ...  
63174  This maybe due to the heavy interlobular conne...  
30371  International labour standards are also an imp...  


In [7]:
train_data.columns
frequency_count = {}

for index, row in train_data.iterrows():
    sense = row[0]
    word = row[1]
    
    frequency_count[word] = {}
    if sense not in frequency_count[word]:
        frequency_count[sense] = 0

    else:
        frequency_count[word][sense]+= 1
    
    print( sense, word )
for sense, count in frequency_count.items():
    print(f"{sense}: {count}")

extend%2:30:02:: extend.v
active%3:00:03:: active.a
serve%2:42:01:: serve.v
case%1:09:00:: case.n
regular%5:00:00:standard:02 regular.a
build%2:36:00:: build.v
life%1:26:01:: life.n
find%2:32:00:: find.v
see%2:31:01:: see.v
case%1:09:00:: case.n
point%1:09:01:: point.n
case%1:11:00:: case.n
point%1:28:00:: point.n
serve%2:42:01:: serve.v
bad%5:00:00:intense:00 bad.a
line%1:04:01:: line.n
hold%2:31:01:: hold.v
common%3:00:02:: common.a
national%3:00:01:: national.a
find%2:39:02:: find.v
common%5:00:00:familiar:02 common.a
case%1:10:02:: case.n
line%1:04:01:: line.n
see%2:32:00:: see.v
life%1:09:00:: life.n
extend%2:30:06:: extend.v
point%1:09:01:: point.n
see%2:31:00:: see.v
keep%2:42:00:: keep.v
extend%2:30:09:: extend.v
see%2:31:03:: see.v
time%1:11:00:: time.n
common%5:00:00:familiar:02 common.a
line%1:04:01:: line.n
point%1:09:01:: point.n
life%1:28:01:: life.n
keep%2:41:03:: keep.v
common%5:00:00:shared:00 common.a
force%1:07:01:: force.n
active%3:00:07:: active.a
major%3:00:07:: m

### Creating a Baseline

Your second task is to create a *baseline* for the task.

A baseline is a "reality check" for a model. Given a very simple heuristic/algorithmic/model solution to the problem, can our neural network perform better than this? Baselines are important as they give us a point of comparison for the actual models. They are commonly used in NLP. Sometimes baseline models are not simple models but previous state-of-the-art.

In this exercise, we will have a simple baseline model that is the "most common sense" (MCS) baseline. For each word form, find the most commonly assigned sense to the word and label a word with that sense. In a fictional dataset, "bank" has two senses: "financial institution," which occurs 5 times, and "side of the river," which occurs 3 times. Thus, all 8 occurrences of "bank" are labeled "financial institution," yielding an MCS accuracy of 5/8 = 62.5%. If a model obtains a higher score than this, we can conclude that the model *at least* is better than selecting the most frequent word sense.

Your task is to write the code for this baseline, train, and test it. The baseline has the knowledge about labels and their frequency only from the train data. You evaluate it on the test data by comparing the ground-truth sense with the one that the model predicts. A good "dumb" baseline in this case is the one that performs quite badly. Expect the model to perform around 0.30 in terms of accuracy. You should use accuracy as your main metric; you can also compute the F1-score.

**[2 marks]**


In [8]:
from sklearn.metrics import accuracy_score

def mcs_baseline(train_data, test_data):

    
    # TRAIN
    
    frequency = {}

    for index, row in train_data.iterrows():
        sense = row[0]
        word = row[1]

        if word not in frequency:
            frequency[word] = {}

        if sense not in frequency[word]:
            frequency[word][sense] = 0

        frequency[word][sense] += 1

    
    # BUILD MODEL
    
    most_common_sense_model = {}

    for word in frequency:
        sense_count = frequency[word]

        max_sense = None
        max_count = -1

        for sense in sense_count:
            if sense_count[sense] > max_count:
                max_count = sense_count[sense]
                max_sense = sense

        most_common_sense_model[word] = max_sense

    
    # TEST
    
    correct_senses = []
    predicted_senses = []

    for index, row in test_data.iterrows():
        true_sense = row[0]
        word = row[1]

        correct_senses.append(true_sense)

        if word in most_common_sense_model:
            predicted_senses.append(most_common_sense_model[word])
        else:
            predicted_senses.append("UNKNOWN")

    
    # ACCURACY
    
    accuracy = accuracy_score(correct_senses, predicted_senses)

    print("Accuracy:", accuracy)

    return most_common_sense_model, accuracy

In [9]:
#for i, (word, sense) in enumerate(list(most_common_sense_model.items())[:10]):
 #   print(word, "->", sense)

model, acc = mcs_baseline(train_data, test_data)
print(acc)

Accuracy: 0.31242603550295855
0.31242603550295855


### Creating Data Iterators

To train a neural network, we first need to prepare the data. This involves converting words (and labels) to a number and organizing the data into batches. We also want the ability to shuffle the examples such that they appear in a random order.

Your task is to create a dataloader for the training and test set you created previously.

You are encouraged to adjust your own dataloader you built for previous assignments. Some things to take into account:

1. Tokenize inputs, keep a dictionary of word-to-IDs and IDs-to-words (vocabulary), fix paddings. You might need to consider doing these for each of the four fields in the dataset.
2. Your dataloader probably has a function to process data. Process each column in the dataset.
3. You might want to clean the data a bit. For example, the first column has some symbols, which might be unnecessary. It is up to you whether you want to remove them and clean this column or keep labels the way they are. In any case, you must provide an explanation of your decision and how you think it will affect the performance of your model. Data and its preprocessing matters, so motivate your decisions.
4. Organize your dataset into batches and shuffle them. You should have something akin to data iterators so that your model can take them.

Implement the dataloader and perform necessary preprocessings.

[**2 marks**]

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

#build vocabulary and encode train sentence
word_to_id = {
    "<PAD>" : 0,
    "<UNK>" : 1
}
encoded_train_sentences = []
train_sense_labels = []

#loop through training data
for index, row in train_data.iterrows():
    sentence_text = row[3].lower()   #sentence column
    sense_label = row[0]       #sense label column
    tokens = sentence_text.split()     # tokenize sentence
#print(tokens)


    sentence_ids = []

    for token in tokens:    
    #add token to vocabulary    
        if token not in word_to_id:
            word_to_id[token] = len(word_to_id)

    #convert token to id
        token_id = word_to_id.get(token, 1)
        sentence_ids.append(token_id)

#store encoded sentence
    encoded_train_sentences.append(sentence_ids)
#store labels
    train_sense_labels.append(sense_label)

print("======== Train data ========")
print("Total train sentence:", len(encoded_train_sentences))
print("Total train labels:", len(train_sense_labels))
print("Vocabulary size:", len(word_to_id))
print("Sample vocabulary", list(word_to_id.items())[:10])
print("First encoded train sentence:", encoded_train_sentences[0])
print("First train label:", train_sense_labels[0])


======== Train data ========
Total train sentence: 60839
Total train labels: 60839
Vocabulary size: 70238
Sample vocabulary [('<PAD>', 0), ('<UNK>', 1), ('(', 2), ('vii', 3), (')', 4), ('bgl', 5), ('-', 6), ('general', 7), ('trust', 8), ('fund', 9)]
First encoded train sentence: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 10, 11, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 2, 26, 4, 27, 6, 28, 29, 8, 9, 10, 30, 29, 31, 32, 33, 34, 35, 36, 34, 11, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 2, 37, 4, 38, 6, 7, 8, 9, 10, 11, 39, 34, 40, 41, 2, 42, 4, 17, 18, 19, 20, 21, 22, 23, 43, 25]
First train label: extend%2:30:02::


In [11]:
#encode test sentences
#test data should not create new vocabulary

encoded_test_sentences = []
test_sense_labels = []

#loop through test data
for index, row in test_data.iterrows():
    sentence_text = row[3].lower()
    sense_label = row[0]
    
    tokens = sentence_text.split()
    sentence_ids = []

    for token in tokens:

        #unknown words 
        token_id = word_to_id.get(token, 1)

        sentence_ids.append(token_id)

    encoded_test_sentences.append(sentence_ids)
    test_sense_labels.append(sense_label)

print(" ============== test data =============")
print("Total test sentences:", len(encoded_test_sentences))
print("Total test labels:", len(test_sense_labels))
print("First encoded test sentence:", encoded_test_sentences[0])
print("First test label:", test_sense_labels[0])

 ============== test data =============
Total test sentences: 15210
Total test labels: 15210
First encoded test sentence: [2090, 433, 6, 3280, 60896, 882, 19, 174, 2585, 17, 116, 17, 5129, 3384, 9578, 53, 12512, 1500, 295, 17, 1, 65, 50, 596, 66, 17, 266, 17, 63, 17, 67, 17, 595, 50, 1497, 979, 53, 298, 91, 3368, 17, 297, 298, 50, 1606, 425, 185, 1680, 68, 940, 2805, 17, 60896, 882, 263, 1694, 2925, 4922, 68]
First test label: lead%2:42:01::


In [12]:
#padding

#finding longest train sentence
max_length = max(len(sentence) for sentence in encoded_train_sentences)

print("=========== padding ============")
print("MAximum sentence length:", max_length)

padded_train_sentences = []

for sentence in encoded_train_sentences:
    #copy sentence
    padded_sent = sentence.copy()

    #add pad tokens (0) until same length
    while len(padded_sent) < max_length:
        padded_sent.append(0)

    padded_train_sentences.append(padded_sent)


#pad test sentences
padded_test_sentences = []

for sentence in encoded_test_sentences:
    #copy sentence
    padded_sent = sentence.copy()

    #add pad tokens (0) until same length
    while len(padded_sent) < max_length:
        padded_sent.append(0)

    padded_test_sentences.append(padded_sent)

print("======== padding for train sentence ==========")

print("First padded train sentence:", padded_train_sentences[0])
print("Length after padding:", len(padded_train_sentences[0]))







=========== padding ============
MAximum sentence length: 281
======== padding for train sentence ==========
First padded train sentence: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 10, 11, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 2, 26, 4, 27, 6, 28, 29, 8, 9, 10, 30, 29, 31, 32, 33, 34, 35, 36, 34, 11, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 2, 37, 4, 38, 6, 7, 8, 9, 10, 11, 39, 34, 40, 41, 2, 42, 4, 17, 18, 19, 20, 21, 22, 23, 43, 25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
#encoded label(sense -> id)
sense_to_id = {}
for label in train_sense_labels:
    if label not in sense_to_id:
        sense_to_id[label] = len(sense_to_id)

encoded_train_labels = []
for label in train_sense_labels:
    label_id = sense_to_id[label]
    encoded_train_labels.append(label_id)

encoded_test_labels = []
for label in test_sense_labels:

    #unseen labels handled 
    if label in sense_to_id:
        encoded_test_labels.append(sense_to_id[label])
    else:
        encoded_test_labels.append(-1)

print(" ============Label Encoding ===========")
print("number of unique labels:", len(sense_to_id))
print("Sample label mapping:", list(sense_to_id.items())[:10])
print("First encoded train label:",encoded_train_labels[0])


 ============Label Encoding ===========
number of unique labels: 222
Sample label mapping: [('extend%2:30:02::', 0), ('active%3:00:03::', 1), ('serve%2:42:01::', 2), ('case%1:09:00::', 3), ('regular%5:00:00:standard:02', 4), ('build%2:36:00::', 5), ('life%1:26:01::', 6), ('find%2:32:00::', 7), ('see%2:31:01::', 8), ('point%1:09:01::', 9)]
First encode train label: 0


In [14]:
#dataset class
class WSDDataset(Dataset):
    def __init__(self, sentences, labels):
        self.sentences = sentences
        self.labels = labels

    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        sentence_tensor = torch.tensor(self.sentences[idx], dtype= torch.long)
        label_tensor  = torch.tensor(self.labels[idx], dtype= torch.long)
        return sentence_tensor, label_tensor
        

In [15]:
#create datasets

train_dataset = WSDDataset(padded_train_sentences, encoded_train_labels)
test_dataset = WSDDataset(padded_test_sentences, encoded_test_labels)


print(" ==========Dataset =======")
print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))
print("First train datset example:", train_dataset[0])

 ==========Dataset =======
Train dataset size: 60839
Test dataset size: 15210
First train datset example: (tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 10, 11, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25,  2, 26,  4, 27,  6, 28, 29,  8,  9, 10,
        30, 29, 31, 32, 33, 34, 35, 36, 34, 11, 15, 16, 17, 18, 19, 20, 21, 22,
        23, 24, 25,  2, 37,  4, 38,  6,  7,  8,  9, 10, 11, 39, 34, 40, 41,  2,
        42,  4, 17, 18, 19, 20, 21, 22, 23, 43, 25,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  

In [16]:
#creating dataloaders

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_laoder = DataLoader(test_dataset, 
                         batch_size = 32,
                         shuffle = False)

In [17]:
print(" ===========Data loader ==========")

for sentence, label in train_loader:
    print("Sentence batch shape:", sentence.shape)
    print("Label batch shape:", label.shape)

    break

 ===========Data loader ==========
Sentence batch shape: torch.Size([32, 281])
Label batch shape: torch.Size([32])


In [18]:
print(len(train_dataset))

60839


In [19]:



def dataloader(path):

    # your code goes here
    # below are only some examples!
    
    def __getitem__(self, idx):
        
        return Y

    def __len__(self):
        
        return X
    
def data_load(something):
    
    return dataloader_train, dataloader_test

# 2.1 LSTM for Word Sense Disambiguation

In this section, we will train an LSTM model to predict word senses based on *contextualized representations*.

You can read more about LSTMs [here](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).


### Model

We will use a **bidirectional** Long Short-Term Memory (LSTM) network to create a representation for the sentences and a **linear** classifier to predict the sense of each word.

As we discussed in the lecture, bidirectional LSTM is using **two** hidden states: one that goes in the left-to-right direction, and another one that goes in the right-to-left direction. PyTorch documentation on LSTMs can be found [here](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html). It says that if the bidirectional parameter is set to True, then "h_n will contain a concatenation of the final forward and reverse hidden states, respectively." Keep it in mind because you will have to ensure that your linear layer for prediction takes input of that size.

When we initialize the model, we need a few things:

1) An embedding layer: a dictionary from which we can obtain word embeddings
2) A LSTM-module to obtain contextual representations
3) A classifier that computes scores for each word-sense given *some* input

The general procedure is the following:

1) For each word in the sentence, obtain word embeddings
2) Run the embedded sentences through the LSTM
3) Select the appropriate hidden state
4) Predict the word-sense 

**Suggestion for efficiency:** *Use a low dimensionality (32) for word embeddings and the LSTM when developing and testing the code, then scale up when running the full training/tests*

Your tasks will be to create **two different models** (both follow the two outlines described above).

-----

Your first model should make a prediction from the LSTM's representation of the target word.

In particular, you run your LSTM on the context in which the target word is used. LSTM will produce a sequence of hidden states. Each hidden state corresponds to a single word from the input context. For example, you should be able to get 37 hidden states for a context that has 37 words/elements in it. Next, take the LSTM's representation of the target word. For example, it can be hidden state number 5, because the fifth word in your context is the target word that you want to predict the meaning for. This target's word representation is the input to your linear layer that makes the final prediction.

**[5 marks]**

In [20]:
class WSDModel_approach1(nn.Module):
    def __init__(self, ...):
        
        # your code goes here
        self.embeddings = ...
        self.rnn = ...
        self.classifier = ...
    
    def forward(self, batch):
        # your code goes here
        
        return predictions

SyntaxError: invalid syntax (2893952739.py, line 2)

Your second model should make a prediction from the final hidden state of your LSTM.

In particular, do the same first steps as in the first approach. But then to make a prediction with your linear layer, you will need to take the last hidden state that your LSTM produces for the whole sequence.

**[5 marks]**

In [ ]:
class WSDModel_approach2(nn.Module):
    def __init__(self, ...):
        # your code goes here
    
    def forward(self, ...):
        # your code goes here
        
        return predictions

### Training and Testing the Model

Now we are ready to train and test our model. What we need now is a loss function, an optimizer, and our data. 

- First, create the loss function and the optimizer.
- Next, iterate over the number of epochs (i.e., how many times we let the model see our data). 
- For each epoch, iterate over the dataset to obtain batches. Use the batch as input to the model, and let the model output scores for the different word senses.
- For each model output, calculate the loss (and print the loss) on the output and update the model parameters.
- Reset the gradients and repeat.
- After all epochs are done, test your trained model on the test set and calculate the total and per-word-form accuracy of your model.

Implement the training and testing of the model.

**[4 marks]**

**Suggestion for efficiency:** *When developing your model, try training and testing the model on one or two batches (for each epoch) of data to make sure everything works! It's very annoying if you train for N epochs to find out that something went wrong when testing the model, or to find that something goes wrong when moving from epoch 0 to epoch 1.*

Do not forget to save your best models as .pickle files. The results should be reproducible for us to evaluate your models.


In [ ]:
train_iter, test_iter, vocab, labels = dataloader(path_to_folder)

loss_function = ...
optimizer = ...
model = ...

for _ in range(epochs):
    # train model
    ...
    
# test model after all epochs are completed

# 2.2 Fine-tuning and Testing BERT for Word Sense Disambiguation

In this section of the lab, you'll try out the transformer, specifically the BERT model. For this, we'll use the Hugging Face library ([https://huggingface.co/](https://huggingface.co/)).

You can find the documentation for the BERT model [here](https://huggingface.co/transformers/model_doc/bert.html) and a general usage guide [here](https://huggingface.co/docs/transformers/quicktour).

What we're going to do is *fine-tune* the BERT model, i.e., update the weights of a pre-trained model. That is, we have a model that is pre-trained on masked language modeling and next sentence prediction (kind of basic, general tasks which are useful for a lot of more specific tasks), but now we apply it to word sense disambiguation with the word representations it has learned.

We'll use the same data splits for training and testing as before, but this time you will use a different dataloader.

Now you create an iterator that collects N sentences (where N is the batch size) then use the BertTokenizer to transform the sentence into integers. For your dataloader, remember to:
* Shuffle the data in each batch
* Make sure you get a new iterator for each *epoch*
* Create a vocabulary of *sense-labels* so you can calculate accuracy 

We then pass this batch into the BERT model (you must have pre-loaded its weights) and update the weights (fine-tune). The BERT model will encode the sentence, then we send this encoded sentence into a prediction layer and collect what it outputs.

As input to the prediction layer, you are free to play with different types of information. For example, the expected way would be to use CLS representation. You can also use other representations and compare them.

About the hyperparameters and training:
* For BERT, usually a lower learning rate works best, between 0.0001-0.000001.
* BERT takes a lot of resources, running it on CPU will take ages, utilize the GPUs :)
* Since BERT takes a lot of resources, use a small batch size (4-8)
* Computing the BERT representation, make sure you pass the mask. It tells the model to ignore padded tokens when computing attention.

**[12 marks]**

In [ ]:
def dataloader_for_bert(path_to_file, batch_size):
    ...

In [ ]:
class BERT_WSD(nn.Module):
    def __init__(self, ...):
        # your code goes here
        self.bert = ...
        self.classifier = ...
    
    def forward(self, batch):
        # your code goes here
        
        return predictions

In [ ]:
loss_function = ...
optimizer = ...
model = ...

for _ in range(epochs):
    # train model
    ...
    
# test model after all epochs are completed

# 3. Evaluation

Explain the difference between the two LSTMs that you have implemented for word sense disambiguation.

Important note: your LSTMs should be nearly the same, but your linear layer must take different inputs. Describe why and how you think this difference will affect the performance of different LSTMs. How does the contextual representation of the whole sequence perform? How does the representation of the target word perform? What is better and for what situations? Why do we observe these differences?

What kind of representations are the different approaches using to predict word senses?

**[4 marks]**

Evaluate your model with per-word form *accuracy* and comment on the results you get. How does the model perform in comparison to the baseline, and how do the models compare to each other? 

Expand on the evaluation by sorting the word-forms by the number of senses they have. Are word forms with fewer senses easier to predict? Give a short explanation of the results you get based on the number of senses per word.

**[4 marks]**

How do the LSTMs perform in comparison to BERT? What's the difference between representations obtained by the LSTMs and BERT?

**[4 marks]**

What could we do to improve all WSD models that we have worked with in this assignment?

**[2 marks]**

# Readings

[1] Kågebäck, M., & Salomonsson, H. (2016). Word Sense Disambiguation using a Bidirectional LSTM. arXiv preprint arXiv:1606.03568.

[2] On WSD: https://web.stanford.edu/~jurafsky/slp3/slides/Chapter18.wsd.pdf

## Your reflections on this lab

Write below your general thoughts, experiences, or reflections on how you worked on this lab.

## Statement of contribution

Briefly state how many times you have met for discussions, who was present, to what degree each member contributed to the discussion and the final answers you are submitting.

## Marks

This assignment has a total of 46 marks.